# Healthcare Access computation

Proportion of the population living within a radius of 5km from a Hhealth Facility - FOSA (+/-60' walking distance)

## 1. Setup

In [ ]:
# Project paths
ROOT_PATH <- '~/workspace'
CODE_PATH <- file.path(ROOT_PATH, 'code')
PROJECT_PATH <- file.path(ROOT_PATH, "pipelines/snt_healthcare_access")
UTILS_PATH <- file.path(PROJECT_PATH, 'utils')

**Validate parameters**

In [ ]:
# Parameters
INPUT_FOSA_FILE <- NULL # Optional file (full path) with health facility locations
WORLDPOP_YEAR <- as.integer(format(Sys.Date(), "%Y")) - 1 # Year for WorldPop raster data

In [ ]:
print(paste0("FOSA file: ", INPUT_FOSA_FILE))
print(paste0("WorldPop population reference year: ", WORLDPOP_YEAR))

In [ ]:
# Global settings
options(scipen=999)
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(UTILS_PATH, "snt_healthcare_access.r"))
setup_ctx <- bootstrap_healthcare_access_context(root_path = ROOT_PATH)

CONFIG_PATH <- setup_ctx$CONFIG_PATH
DATA_PATH <- setup_ctx$DATA_PATH
OUTPUT_DATA_PATH <- setup_ctx$OUTPUT_DATA_PATH
OUTPUT_PLOTS_PATH <- setup_ctx$OUTPUT_PLOTS_PATH
INTERMEDIATE_RESULTS_PATH <- setup_ctx$INTERMEDIATE_RESULTS_PATH

reticulate::py_config()$python

# Load SNT config
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, "SNT_config.json")) },
                        error = function(e) {
                          msg <- paste0("Error while loading configuration", conditionMessage(e))
                          cat(msg)
                          stop(msg)
                        })

pipeline_msg(glue("SNT configuration loaded from: {file.path(CONFIG_PATH, 'SNT_config.json')}"))

In [ ]:
# Set variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ORG_UNITS_LEVEL <- config_json$SNT_CONFIG$ANALYTICS_ORG_UNITS_LEVEL
print(paste("Country code: ", COUNTRY_CODE))

# Global variables
admin_col <- "ADM2_ID"
country_epsg_degrees <- 4326 # for plotting
country_epsg_meters <- 32630 # for creating the buffer areas

# column names
latitude_col <- "LATITUDE"
longitude_col <- "LONGITUDE"
coordinate_cols <- c(longitude_col, latitude_col) # longitude (x) first, latitude (y) second
status_closed_col <- "CLOSED_DATE" # for FOSA status 

## 2. Load data

### 2.1. Load spatial administrative unit data

In [ ]:
# load as vector data
dhis2_formatted_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
spatial_units_data <- load_spatial_units_data(
  shapes_file = NA,
  dhis2_dataset = dhis2_formatted_dataset,
  country_code = COUNTRY_CODE
)

Make the related data objects: the admin data and the country polygon

In [ ]:
prepared_spatial <- prepare_spatial_admin_objects(
  spatial_units_data = spatial_units_data,
  country_epsg_degrees = country_epsg_degrees
)

spatial_units_data <- prepared_spatial$spatial_units_data
admin_data <- prepared_spatial$admin_data
all_country <- prepared_spatial$all_country

### 2.2. Load population data

In [ ]:
# Import population raster, according to the reference year; format is {COUNTRY_CODE}_pop_{WORLDPOP_YEAR}_CN_100m_*.tif
wpop_raw_path <- file.path(DATA_PATH, "worldpop", "rasters")

if (!dir.exists(wpop_raw_path)) {
  stop(glue(
    "The {wpop_raw_path} directory, for WorldPop raster data, is missing."
  ))
}

wpop_pattern <- sprintf("%s_pop_%s_CN_100m_[[:alnum:]_-]+\\.tif$", tolower(COUNTRY_CODE), as.character(WORLDPOP_YEAR))
matching_files <- list.files(path = wpop_raw_path, pattern = wpop_pattern)

pop_path <- file.path(wpop_raw_path, matching_files[1])

pop_data <- tryCatch(
rast(pop_path),
error = function(e) stop(glue("Error while loading population raster: {conditionMessage(e)}"))
)
pipeline_msg(glue("Population raster data loaded: {pop_path}"))

### 2.3. Load data to use for FOSA locations

Import the locations (points) of the healthcare units.  
-If input file (`FOSA_FILE`) is not provided, load available DHIS2 pyramid data.

In [ ]:
# Check, if user inputs data, that the user's data is usable
fosa_dt <- import_fosa_data(
    input_file_path=INPUT_FOSA_FILE,
    pipeline_dhis2_dataset=dhis2_formatted_dataset,
    pipeline_country_code=COUNTRY_CODE,
    latitude_colname="LATITUDE",
    longitude_colname="LONGITUDE")

setDT(fosa_dt)

## 3. FOSA data pre-processing  
  
Determine active Health facilities:  
    -Remove closed `Health facilities` from the FOSA table using `closing date` column (if available).  

Clean coordinates:  
    -Remove `Health facilities` with incomplete coordinates (Lon, Lat) from the FOSA table.

In [ ]:
# format the latitude and longitude columns as expected by the notebook
input_longitude_col <- grep(glue::glue("^{longitude_col}$"), names(fosa_dt), ignore.case = TRUE, value = TRUE)
input_latitude_col <- grep(glue::glue("^{latitude_col}$"), names(fosa_dt), ignore.case = TRUE, value = TRUE)
setnames(fosa_dt, old=c(input_longitude_col, input_latitude_col), new=c(longitude_col, latitude_col))
tolower(names(fosa_dt))

### 3.1. Remove health facilities with **CLOSED_DATE** (if present)

In [ ]:
# if the data contains a column indicated the date when closed, filter only units which were not closed
if(any(tolower(names(fosa_dt)) == tolower(status_closed_col))) {
    initial_rows <- nrow(fosa_dt)
    fosa_dt <- fosa_dt[is.na(get(status_closed_col)),]
    filtered_rows <- nrow(fosa_dt)
    removed_rows <- initial_rows - filtered_rows
    pipeline_msg(glue("Removed {removed_rows} observations, FOSA which are no longer in operation."))
}

### 3.2. Specific case for **Niger (NER)**, we include two extra filters for health facilities:  
  
#### 3.2.1 Remove closed health facilities from the FOSA table where the name contains `(clôture)` or `(fermé)`.  
  
  -Compare strings with ignore case and accents removed.

In [ ]:
# Remove any HF that have these strings in the name together with 
if (COUNTRY_CODE == "NER") {
    closing_suffixes <- c("cloture" , "ferme")
    ou_column_name <- glue("LEVEL_{ORG_UNITS_LEVEL}_NAME")
    
    # Logical vector to select rows
    remove_rows <- str_detect(fosa_dt[[ou_column_name]], regex(paste0(closing_suffixes, collapse = "|"), ignore_case = TRUE))
    rows_removed <- fosa_dt[remove_rows, ]
    fosa_dt <- fosa_dt[!remove_rows, ]
    
    print(glue("FOSA Dimensions: {paste(dim(fosa_dt), collapse=',')}"))
    
    # removed rows:
    rows_removed
}

#### 3.2.2 Remove closed `Health facilities` from the FOSA table where the name is part of the `Structure Clôturées` Organisation units group.  

In [ ]:
# Use the list org units group "Structures Clôturées" that contains the "Closed" health facilitie ids

ou_groups <- file.path(DATA_PATH, "dhis2/extracts_raw/organisation_unit_groups/NER_organisation_unit_groups.parquet")
if (COUNTRY_CODE == "NER" & file.exists(ou_groups)) {
    pipeline_msg(glue("Filtering NER organisation units with: {ou_groups}"))
    org_units <- read_parquet(ou_groups)
    pipeline_msg(glue("FOSA table dimensions: {paste(dim(fosa_dt), collapse=', ')}"))
    
    # Select list: Structures Clôturées (oshkuclYJAw)    
    structures_cloturees <- org_units[org_units$id == "oshkuclYJAw", ]$organisation_units[[1]]
    if (length(structures_cloturees) > 0) {
        ou_column_id <- glue("LEVEL_{ORG_UNITS_LEVEL}_NAME")
        # Filter fosa_dt: keep rows where the value is NOT in structures_cloturees
        fosa_dt <- fosa_dt[!(fosa_dt[[ou_column_id]] %in% structures_cloturees), ]
    }
 
    pipeline_msg(glue("FOSA list filtered using organisation units list Structures Clôturées, FOSA table dimensions: {paste(dim(fosa_dt), collapse=', ')}"))
}

### 3.3. Remove units which are missing either latitude or longitude

In [ ]:
# filter and select coordinate_cols
fosa_dt <- unique(
    fosa_dt[
        (!is.na(get(longitude_col))) & (!is.na(get(latitude_col))),
        .SD,
        .SDcols = c(coordinate_cols)
        ]
    )
print(dim(fosa_dt))

In [ ]:
# Convert the FOSA data table to an sf object using the specified coordinate columns.
fosa_vect <- st_as_sf(fosa_dt, coords = coordinate_cols, crs = country_epsg_degrees)
fosa_vect_filtered <- sf::st_filter(fosa_vect, all_country, .predicate = st_within)
pipeline_msg(glue("Using {nrow(fosa_vect)} distinct observations, which have geographic coordinates within the country boundaries."))

fosa_vect_filtered_filename <- glue("{COUNTRY_CODE}_FOSA_filtered.gpkg")
write_sf(fosa_vect_filtered, file.path(INTERMEDIATE_RESULTS_PATH, fosa_vect_filtered_filename))

In [ ]:
# free up resources
rm(fosa_vect)
rm(all_country)
rm(fosa_dt)
gc()

## 4. Start health access computation

In [ ]:
# make the circles around each health unit
overlapping_coverage_vect <- make_coverage_radii_sf(
  input_vect = fosa_vect_filtered,
  coordinate_colnames = coordinate_cols,
  epsg_value_degrees = country_epsg_degrees,
  epsg_value_meters = country_epsg_meters,
  radius_meters = 5000
)

In [ ]:
# dissolve everything into one multipolygon
coverage_vect <- st_union(overlapping_coverage_vect)
coverage_vect <- st_as_sf(coverage_vect)

coverage_vect_filename <- glue("{COUNTRY_CODE}_health_coverage_buffers.gpkg")
write_sf(coverage_vect, file.path(INTERMEDIATE_RESULTS_PATH, coverage_vect_filename))

In [ ]:
# free up resources
rm(overlapping_coverage_vect)
rm(fosa_vect_filtered)
gc()

In [ ]:
pipeline_msg("Computing 5 km radii around each healthcare unit. This will take a few minutes..")
    
# determine which cells are included in at least one of the radii
pop_healthcare_rast <- make_rasterized_inclusion_data(
  buffer_vect = coverage_vect,
  raster_data = pop_data,
  epsg_value_degrees = country_epsg_degrees,
  value_inside = 1,
  value_outside = 0
)

pipeline_msg("Radii around each healthcare unit done.")

In [ ]:
# inject the dummy variable into the population data
pop_healthcare_data <- c(pop_data, pop_healthcare_rast)
names(pop_healthcare_data) <- c("POP_TOTAL", "COVERED")

In [ ]:
# free up resources
rm(coverage_vect)
rm(pop_healthcare_rast)
rm(pop_data)
gc()

In [ ]:
pipeline_msg("Computing the population inside the 5 km buffers. This is computationally intensive and will take a few minutes..")

# Compute and write to disk (chunked, memory-safe)
pop_covered_healthcare <- writeRaster(
  pop_healthcare_data$POP_TOTAL * pop_healthcare_data$COVERED,
  file.path(DATA_PATH, "healthcare_access/.pop_covered_healthcare.tif"),
  overwrite = TRUE
)

names(pop_covered_healthcare) <- "POP_COVERED"
pipeline_msg("Population buffer counts done.")

In [ ]:
# free up resources
gc()

In [ ]:
# convert spatial_units_data from sf to terra SpatVector for rasterization
spatial_units_vect <- terra::vect(spatial_units_data)

pipeline_msg("Rasterizing the spatial units, based on the population grid.")
# rasterize using the ID column stored in admin_col
adm2_raster <- terra::rasterize(
  spatial_units_vect,
  pop_healthcare_data$POP_TOTAL,   # template raster
  field = admin_col                # use ID column for zones
)
pipeline_msg("Rasterization done.")

In [ ]:
output_df <- compute_population_by_admin(
  pop_total_raster = pop_healthcare_data$POP_TOTAL,
  pop_covered_raster = pop_covered_healthcare,
  adm_raster = adm2_raster,
  admin_col = admin_col,
  admin_data = admin_data
)

## 5. Save output results

In [ ]:
# write to file
output_df_filename_stem <- glue("{COUNTRY_CODE}_population_covered_health")
fwrite(output_df, file.path(OUTPUT_DATA_PATH, glue("{output_df_filename_stem}.csv")))
write_parquet(output_df, file.path(OUTPUT_DATA_PATH, glue("{output_df_filename_stem}.parquet")))
pipeline_msg(glue("Health access coverage saved: {file.path(OUTPUT_DATA_PATH, glue('{output_df_filename_stem}.parquet'))}"))